### **Exercise: Sentiment Analysis and Key Insights Extraction from Ford Car Reviews**

### **Problem Statement:**
You have been provided with a dataset containing Ford car reviews. Your task is to use LangChain and the concepts you’ve learned to perform the following tasks:

1. **Sentiment Analysis**: Analyze the sentiment of each review, categorize it as positive, neutral, or negative, and store the result.
2. **Key Insights Extraction**: Extract key pieces of information from each review, such as the pros and cons mentioned, and the specific features the reviewer liked or disliked (e.g., vehicle performance, comfort, price).

You will build a LangChain-based solution that leverages language models to automatically extract this information and provide a structured summary of the reviews. 

---
### **Steps to Solve:**

#### **Step 1: Load the Dataset**
- The dataset file is named `ford_car_reviews.csv` and is sourced from Kaggle: [Edmunds Consumer Car Ratings and Reviews](https://www.kaggle.com/datasets/ankkur13/edmundsconsumer-car-ratings-and-reviews).
- For this exercise, **limit the data to the first 25 records**. This can be achieved by using `df.head(25)` or `df.iloc[:25]` when loading the data into a DataFrame.

#### **Step 2: Define the Sentiment Analysis Task**
- Use LangChain to create a pipeline to classify the sentiment of each review.
- Define prompts that can guide the model to evaluate the sentiment. For example:
  - "Given the following car review, classify the sentiment as positive, neutral, or negative."

#### **Step 3: Key Insights Extraction**
- Use LangChain to create a pipeline to extract pros, cons, and notable features from each review. Define prompts such as:
  - "What are the pros and cons of the vehicle described in the following review?"
  - "What specific features of the vehicle does the reviewer like or dislike?"

#### **Step 4: Update the DataFrame with New Information**
- Run the pipeline for each review and collect the sentiment and insights.
- Once the analysis and extraction are complete, update the original DataFrame with additional columns to include:
  - Sentiment (positive, neutral, negative)
  - Pros
  - Cons
  - Liked_Features
  - Disliked_Features

---

### **Example Output:**

```json
{
  "Review_Date": "03/07/13",
  "Vehicle_Title": "2006 Ford Mustang Coupe",
  "Review_Text": "With the expected arrival of our 6th child...",
  "Rating": 4.125,
  "Sentiment": "Positive",
  "Pros": "Good driving experience, Large seating capacity, Great options",
  "Cons": "None mentioned",
  "Liked_Features": ["Driving experience", "Seating capacity", "Options available"],
  "Disliked_Features": []
}
```

#STEP 1#

In [9]:
import os,json,re,getpass
from dotenv import load_dotenv

load_dotenv(override=True)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API KEY:")

In [10]:
import pandas as pd

df = pd.read_csv("ford_car_reviews.csv",nrows=25)

print(df)

    Unnamed: 0                  Review_Date    Author_Name  \
0            0   on 06/06/18 14:19 PM (PDT)         Vicki    
1            1   on 08/12/17 06:06 AM (PDT)           Tom    
2            2   on 06/15/17 05:43 AM (PDT)           Ray    
3            3   on 05/18/17 17:33 PM (PDT)    Don Watson    
4            4   on 01/03/16 18:03 PM (PST)     One owner    
5            5   on 10/24/15 12:40 PM (PDT)          Adam    
6            6   on 10/29/11 04:57 AM (PDT)       amos247    
7            7   on 07/25/11 12:15 PM (PDT)      dave3012    
8            8   on 07/21/11 11:28 AM (PDT)      ronnzy98    
9            9   on 12/06/10 00:00 AM (PST)     Anonymous    
10          10   on 10/29/10 00:00 AM (PDT)           Stu    
11          11   on 08/22/10 17:37 PM (PDT)        shackm    
12          12   on 08/20/10 10:34 AM (PDT)          Chet    
13          13   on 07/22/10 15:00 PM (PDT)     Max Lives    
14          14   on 06/19/10 10:05 AM (PDT)         kells    
15      

In [11]:
#STEP 2: Sentiment Analysis Task
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq")


In [12]:
llm_response = llm.invoke("How is DON?")
llm_response

AIMessage(content='I’m not sure which “DON” you’re referring to. Could you tell me a bit more—are you asking about a person named Don, an organization, a project, or something else? That’ll help me give you a useful answer.', additional_kwargs={'reasoning_content': 'The user asks: "How is DON?" Likely they are asking about "DON" maybe a person, or "DON" abbreviation. Could be "Department of the Navy"? Or "DON" as a name. The user might be asking for a status or health of someone named Don. There\'s no context. We need to respond politely asking for clarification, maybe ask who is Don? Or if they mean "Department of Nutrition"? The instruction: we must follow policies. There\'s no disallowed content. So we can ask for clarification.'}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 75, 'total_tokens': 243, 'completion_time': 0.352661598, 'prompt_time': 0.004552289, 'queue_time': 0.343458979, 'total_time': 0.357213887, 'completion_tokens_details': {'reasoni

In [15]:
from langchain_core.messages import HumanMessage, SystemMessage

system_prompt = """
You are a professional Sentiment and Car Review Analyst.

Analyze the Ford car review provided by the customer.

Work on the following points:
1. Classify the sentiment as positive, neutral, or negative.
2. Identify the pros and cons of the vehicle.
3. Identify the specific features the reviewer likes or dislikes.

Return a structured JSON object with exactly these five attributes:

{
    "Sentiment": "positive/negative/neutral",
    "Pros": [],
    "Cons": [],
    "Liked_Features": [],
    "Disliked_Features": []
}

Return ONLY valid JSON.
"""

def analyze_review(review):

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Car Review:\n{review}")
    ]

    response = llm.invoke(messages)

    return json.loads(response.content)

df["analysis"] = df["Review"].apply(analyze_review)


In [31]:
df['analysis'][1]

{'Sentiment': 'positive',
 'Pros': ['Engine is fine and sounds good',
  'Great mileage',
  'Good power',
  'Attractive, hot-looking car',
  'Fun to drive'],
 'Cons': ['Transmission is very finicky and difficult to master',
  'Gear changes are harsh and can cause lurching',
  'Ride is harsh with a lot of road noise',
  'Requires a lot of finesse to avoid stalling or over‑revving'],
 'Liked_Features': ['V6 engine performance',
  'Fuel efficiency',
  'Power output',
  'Styling and appearance',
  'Driving enjoyment'],
 'Disliked_Features': ['Transmission feel and responsiveness',
  'Gear shift smoothness',
  'Ride comfort and road noise']}

In [35]:
import json

df['sentiment'] = df['analysis'].apply(lambda x: x['Sentiment'])
df['pros'] = df['analysis'].apply(lambda x: x['Pros'])
df['cons'] = df['analysis'].apply(lambda x: x['Cons'])
df['liked_features'] = df['analysis'].apply(lambda x: x['Liked_Features'])
df['disliked_features'] = df['analysis'].apply(lambda x: x['Disliked_Features'])


In [36]:
df

,Unnamed: 0,Review_Date,Author_Name,Vehicle_Title,Review_Title,Review,Rating,analysis,sentiment,pros,cons,liked_features,disliked_features
0,0,on 06/06/18 14:19 PM (PDT),Vicki,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,2006 Mustang GT,Doesn’t disappoint,5.000,"{'Sentiment': 'positive', 'Pros': ['Overall sa...",positive,[Overall satisfaction],[],[],[]
1,1,on 08/12/17 06:06 AM (PDT),Tom,2006 Ford Mustang Coupe V6 Standard 2dr Coupe ...,DREAM CAR,I bought mine 4/17 with 98K. Have been wantin...,3.000,"{'Sentiment': 'positive', 'Pros': ['Engine is ...",positive,"[Engine is fine and sounds good, Great mileage...",[Transmission is very finicky and difficult to...,"[V6 engine performance, Fuel efficiency, Power...","[Transmission feel and responsiveness, Gear sh..."
2,2,on 06/15/17 05:43 AM (PDT),Ray,2006 Ford Mustang Coupe V6 Premium 2dr Coupe (...,Great Ride,There will always be a 05-09 mustang for sale...,5.000,"{'Sentiment': 'positive', 'Pros': ['Reasonably...",positive,"[Reasonably priced (fairly reasonable), Good i...",[],"[Affordable price, Strong resale/investment po...",[]
3,3,on 05/18/17 17:33 PM (PDT),Don Watson,2006 Ford Mustang Coupe V6 Deluxe 2dr Coupe (4...,I have wanted a Mustang for 40 years.,I bought my car from an auction I work at ( A...,5.000,"{'Sentiment': 'positive', 'Pros': ['Powerful V...",positive,"[Powerful V6 engine, Air Aid cold air injector...",[],"[V6 engine, Cold air intake (Air Aid), Throttl...",[]
4,4,on 01/03/16 18:03 PM (PST),One owner,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,One owner,I bought this car spankin new and i still am ...,5.000,"{'Sentiment': 'positive', 'Pros': ['Excellent ...",positive,"[Excellent handling – the car hugs the road, H...","[Had to replace the alternator, Normal wear it...","[Road-hugging handling, Instant responsiveness...","[Alternator failure, Regular wear on tires and..."
5,5,on 10/24/15 12:40 PM (PDT),Adam,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,Poor,"Lots of problems with Ford these days, sensor...",3.000,"{'Sentiment': 'negative', 'Pros': [], 'Cons': ...",negative,[],"[sensor issues, cam phaser problems, solenoid ...",[],"[sensors, cam phasers, solenoid]"
6,6,on 10/29/11 04:57 AM (PDT),amos247,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,06 mustang gt with a few mods,Bought mine used 20k on it and have added a S...,4.625,"{'Sentiment': 'positive', 'Pros': ['Great bang...",positive,"[Great bang‑for‑buck performance, Decent fuel ...",[],"[SCT tuner, Ford CAI (engine tuning), Flowmast...",[]
7,7,on 07/25/11 12:15 PM (PDT),dave3012,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,2006 Ford Mustang GT Premium 2dr Coupe (4.6L 8...,I bought my preowned 06 a few weeks back and ...,4.375,"{'Sentiment': 'positive', 'Pros': ['Upgraded A...",positive,"[Upgraded Airaid air filter system, Flowmaster...",[],"[Airaid air filter, Flowmaster exhaust, High p...",[]
8,8,on 07/21/11 11:28 AM (PDT),ronnzy98,2006 Ford Mustang Coupe GT Deluxe 2dr Coupe (4...,Get Rid of it before 100K,I drive 50 miles each way to work and traded ...,3.500,"{'Sentiment': 'negative', 'Pros': [], 'Cons': ...",negative,[],[Engine knocking noise and significant loss of...,[],"[Cam phaser, Engine durability, Repair expense]"
9,9,on 12/06/10 00:00 AM (PST),Anonymous,2006 Ford Mustang Coupe GT Premium 2dr Coupe (...,Get this car.,This car is just awesome. The 4.6L V8 makes ...,4.625,"{'Sentiment': 'positive', 'Pros': ['Powerful 4...",positive,"[Powerful 4.6L V8 engine, Exciting stock exhau...",[],"[4.6L V8 powertrain, Stock exhaust sound, Reli...",[]
